In [ ]:
import numpy as np
import copy

In [ ]:

# Estimador Base fornecido pronto: Toco de Decisao (Decision Stump) em NumPy
class StumpDidatico:
    def __init__(self):
        self.feat, self.thresh, self.left_class, self.right_class = 0, 0.0, 0, 0

    def fit(self, X, y, sample_weight=None):
        X, y = np.asarray(X), np.asarray(y)
        w = np.ones(len(y)) / len(y) if sample_weight is None else np.asarray(sample_weight)
        classes, best_err = np.unique(y), 2.0
        for f in range(X.shape[1]):
            for t in np.unique(X[:, f]):
                left_mask = X[:, f] <= t
                for c_l in classes:
                    for c_r in classes:
                        preds = np.where(left_mask, c_l, c_r)
                        err = np.sum(w * (preds != y)) / np.sum(w)
                        if err < best_err:
                            best_err, self.feat, self.thresh, self.left_class, self.right_class = err, f, t, c_l, c_r
        return self

    def predict(self, X):
        X = np.asarray(X)
        return np.where(X[:, self.feat] <= self.thresh, self.left_class, self.right_class)


class SimpleMultiClassBoosting:
    def __init__(self, base_estimator, n_estimators=10):
        self.base_estimator = base_estimator
        self.n_estimators = n_estimators
        self.learners = []
        self.learner_weights = []

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        self.learners = []
        self.learner_weights = []

        # 1. Inicializa os pesos uniformemente
        sample_weights = np.full(X.shape[0], 1.0 / X.shape[0])

        for _ in range(self.n_estimators):
            learner = copy.deepcopy(self.base_estimator)
            learner.fit(X, y, sample_weight=sample_weights)
            y_pred = learner.predict(X)

            incorrect = (y_pred != y)
            # 2. Calcula a taxa de erro ponderada
            learner_error = np.average(incorrect, weights=sample_weights)

            if learner_error == 0:
                self.learners.append(learner)
                self.learner_weights.append(1.0)
                break
            if learner_error >= 1.0 - (1.0 / n_classes):
                break

            # 3. Calcula o peso do modelo via fórmula SAMME
            learner_weight = np.log((1.0 - learner_error) / learner_error) + np.log(n_classes - 1)
            
            self.learners.append(learner)
            self.learner_weights.append(learner_weight)

            # Atualiza e normaliza os pesos das amostras
            sample_weights *= np.exp(learner_weight * incorrect)
            sample_weights /= np.sum(sample_weights)

        return self

    def predict(self, X):
        X = np.asarray(X)
        learner_preds = np.array([learner.predict(X) for learner in self.learners])
        weighted_preds = np.zeros((X.shape[0], len(self.classes_)))
        for i, c in enumerate(self.classes_):
            # 4. Soma os pesos ponderados dos modelos que votaram na classe 'c'
            weighted_preds[:, i] = np.dot(self.learner_weights, (learner_preds == c))
            
        return self.classes_[np.argmax(weighted_preds, axis=1)]